In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import sys
sys.path.append('..')

from tools.geometry import generate_detector
from tools.utils import generate_random_point_inside_cylinder, print_propagation_params
from tools.losses import compute_softmin_loss, WC_loss
from tools.simulation import setup_event_simulator

import jax
import jax.numpy as jnp
from jax import grad, jit, value_and_grad
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import time
import os

# Import BO-LEAP functions from the optimization module
from tools.optimization.bo_leap import setup_and_run_bo_leap

# Create plots folder if it doesn't exist
if not os.path.exists('../plots'):
    os.makedirs('../plots')
    print('Created ../plots folder')

In [ ]:
# Setup detector and simulation parameters
default_json_filename = '../config/IWCD_geom_config.json'

detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

# Simulation parameters
Nphot = 1_500_000
K = 6

# Temperature parameters
TRUE_TAU_GS = 0.001  # Used for generating true data
SIM_TAU_GS = 0.01  # Used during optimization

# Parameter bounds for normalization
PARAM_MIN = jnp.array([2.0, 0.0, 2.0])  # [scatter_length, reflection_rate, absorption_length]
PARAM_MAX = jnp.array([6.0, 1.0, 10.0])  # [scatter_length, reflection_rate, absorption_length]

# Setup event simulator
simulate_event = setup_event_simulator(default_json_filename, Nphot, temperature=None, K=K, is_calibration=True)

# BO-LEAP hyperparameters
BO_LEAP_ITERATIONS = 2 # Number of BO-LEAP GP iterations/resets
BO_LEAP_LOCAL_STEPS = 70 # Local optimization steps per iteration
BO_LEAP_K = 10  # Population size
BO_LEAP_J = 10  # Gradient descent steps
BO_LEAP_ALPHA = 0.01 # Step size
BO_LEAP_M = 100  # Max points for GP subset

## Application-specific Functions

In [ ]:
def create_loss_function_for_boleap(source_params, true_data, key):
    """Create loss function for BO-LEAP that works in original parameter space."""

    @jit
    def loss_fn(params_3d):
        # Add tau_gs to create full detector params
        detector_params = jnp.concatenate([params_3d, jnp.array([SIM_TAU_GS])])

        # Simulate event
        simulated_data = simulate_event(source_params, detector_params, key)

        # Compute loss
        return WC_loss(
                detector_points, *true_data, *simulated_data,
                lambda_poisson=1.0,
                lambda_time=0.0
            )

    return loss_fn

In [ ]:
def run_single_calibration_boleap(true_params, initial_guess, source_params, true_data, key, case_name=""):
    """Run a single calibration case using BO-LEAP."""

    print(f"\n{'=' * 60}")
    print(f"Running: {case_name}")
    print(f"{'=' * 60}")

    # Display true parameters
    print(f"\nTrue parameters:")
    print(f"  Scatter Length: {true_params[0]:.4f} m")
    print(f"  Reflection Rate: {true_params[1]:.4f}")
    print(f"  Absorption Length: {true_params[2]:.4f} m")

    print(f"\nInitial guess:")
    print(f"  Scatter Length: {initial_guess[0]:.4f} m")
    print(f"  Reflection Rate: {initial_guess[1]:.4f}")
    print(f"  Absorption Length: {initial_guess[2]:.4f} m")

    # Create loss function
    loss_fn = create_loss_function_for_boleap(source_params, true_data, key)

    # Run BO-LEAP optimization
    start_time = time.time()

    best_params, best_loss, history = setup_and_run_bo_leap(
        loss_fn_original=loss_fn,
        lower_bounds=PARAM_MIN,
        upper_bounds=PARAM_MAX,
        initial_guess=initial_guess,
        n_iterations=BO_LEAP_ITERATIONS,
        local_steps=BO_LEAP_LOCAL_STEPS,
        K=BO_LEAP_K,
        J=BO_LEAP_J,
        M=BO_LEAP_M,
        alpha=BO_LEAP_ALPHA,
        key=key
    )

    runtime = time.time() - start_time

    # Calculate errors
    param_errors = jnp.abs(best_params - true_params)
    mse = jnp.mean(param_errors ** 2)

    # Print results
    print(f"\nOptimization completed in {runtime:.2f} seconds")
    print(f"\nFinal parameters:")
    print(f"  Scatter Length: {best_params[0]:.4f} m (error: {param_errors[0]:.4f})")
    print(f"  Reflection Rate: {best_params[1]:.4f} (error: {param_errors[1]:.4f})")
    print(f"  Absorption Length: {best_params[2]:.4f} m (error: {param_errors[2]:.4f})")
    print(f"\nMSE: {mse:.6f}")
    print(f"Final Loss: {best_loss:.6f}")

    return {
        'final_params': best_params,
        'final_loss': best_loss,
        'history': history,  # Full BO-LEAP history
        'param_errors': param_errors,
        'mse': float(mse),
        'runtime': runtime,
        'true_params': true_params,
        'initial_guess': initial_guess
    }

In [ ]:
def create_test_cases():
    """Create 4 different test cases with varying conditions."""
    test_cases = []

    # Case 1: Easy case - initial guess close to true values
    case1 = {
        'name': 'Easy Case: Close Initial Guess',
        'true_params': jnp.array([4.0, 0.2, 6.0]),
        'initial_guess': jnp.array([3.8, 0.19, 5.8]),
        'color': 'blue'
    }
    test_cases.append(case1)

    # Case 2: Medium case - moderate offset
    case2 = {
        'name': 'Medium Case: Moderate Offset',
        'true_params': jnp.array([3.5, 0.25, 7.0]),
        'initial_guess': jnp.array([4.5, 0.18, 5.5]),
        'color': 'green'
    }
    test_cases.append(case2)

    # Case 3: Hard case - large offset
    case3 = {
        'name': 'Hard Case: Large Offset',
        'true_params': jnp.array([5.0, 0.15, 5.0]),
        'initial_guess': jnp.array([3.0, 0.35, 8.0]),
        'color': 'red'
    }
    test_cases.append(case3)

    # Case 4: Very hard case - extreme offset
    case4 = {
        'name': 'Very Hard Case: Extreme Offset',
        'true_params': jnp.array([3.0, 0.35, 7.5]),
        'initial_guess': jnp.array([5.8, 0.15, 3.2]),
        'color': 'purple'
    }
    test_cases.append(case4)

    return test_cases

## Visualization Functions

In [ ]:
def plot_boleap_loss_progression(results_list, test_cases):
    """Plot loss progression for BO-LEAP optimization with percentile-based y-axis limits."""

    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, (results, case) in enumerate(zip(results_list, test_cases)):
        ax = axes[idx]
        history = results['history']
        color = case['color']

        # Collect all gradient losses and their evaluation indices
        all_grad_losses = []
        all_grad_indices = []

        for iter_data in history['iterations']:
            iter_idx = iter_data['iteration']

            # Use the stored evaluation indices and losses
            grad_indices = iter_data['valid_s_eval_indices']
            grad_losses = iter_data['valid_s_losses']

            # Plot this iteration's gradient descent
            if len(grad_losses) > 0:
                # Convert to numpy for easier handling
                grad_losses = np.array(grad_losses)
                grad_indices = np.array(grad_indices)

                # Sample for display if too many points
                if len(grad_losses) > 50:
                    sample_step = len(grad_losses) // 50
                    sample_indices = np.arange(0, len(grad_losses), sample_step)
                    sampled_losses = grad_losses[sample_indices]
                    sampled_indices = grad_indices[sample_indices]
                else:
                    sampled_losses = grad_losses
                    sampled_indices = grad_indices

                # Plot with iteration color
                iter_color = plt.cm.viridis(iter_idx / (BO_LEAP_ITERATIONS - 1) if BO_LEAP_ITERATIONS > 1 else 0.5)
                ax.plot(sampled_indices, sampled_losses,
                        'o-', color=iter_color, markersize=3, linewidth=1,
                        alpha=0.7, label=f'Iter {iter_idx + 1}')

            all_grad_losses.extend(grad_losses)
            all_grad_indices.extend(grad_indices)

        # Plot best-so-far line
        if all_grad_losses:
            all_grad_losses = np.array(all_grad_losses)
            all_grad_indices = np.array(all_grad_indices)

            # Sort by evaluation index
            sort_idx = np.argsort(all_grad_indices)
            all_grad_indices = all_grad_indices[sort_idx]
            all_grad_losses = all_grad_losses[sort_idx]

            # Compute best-so-far using numpy's cumulative minimum
            best_so_far = np.minimum.accumulate(all_grad_losses)

            ax.plot(all_grad_indices, best_so_far,
                    'k-', linewidth=2, label='Best found', alpha=0.8, zorder=10)

            # Mark overall best
            best_idx = np.argmin(all_grad_losses)
            ax.scatter(all_grad_indices[best_idx], all_grad_losses[best_idx],
                       c='yellow', s=100, marker='*', edgecolors='black',
                       linewidth=1.5, zorder=11)

            # Set y-axis limits based on percentiles to handle outliers
            y_min = np.min(all_grad_losses)
            y_max = np.percentile(all_grad_losses, 90)  # Use 90th percentile for upper limit
            # Add some padding
            y_range = y_max - y_min
            ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.05 * y_range)

        # Add iteration boundaries
        for iter_data in history['iterations']:
            boundary = iter_data['end_idx']
            ax.axvline(x=boundary, color='gray', linestyle='--',
                       alpha=0.3, linewidth=0.8)

        # Format subplot
        ax.set_xlabel('Evaluation Index', fontsize=10)
        ax.set_ylabel('Loss', fontsize=10)
        ax.set_title(case['name'], fontsize=11, fontweight='bold')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.2)
        ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

        # Add final loss text
        final_loss_text = f"Final: {results['final_loss']:.6f}"
        ax.text(0.02, 0.98, final_loss_text,
                transform=ax.transAxes, fontsize=9,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.suptitle(f'BO-LEAP Optimization Progress (α={BO_LEAP_ALPHA:.1e})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../plots/detector_calibration_boleap_progress.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_parameter_evolution_boleap(results_list, test_cases):
    """Plot parameter evolution during BO-LEAP optimization with distance from minimum loss."""

    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    param_names = ['Scatter Length (m)', 'Reflection Rate', 'Absorption Length (m)']

    # First subplot: Distance from minimum loss
    ax = axes[0, 0]
    for results, case in zip(results_list, test_cases):
        history = results['history']
        all_losses = np.array(history['all_y'])  # Convert to numpy

        # Compute best loss found so far at each evaluation using numpy
        best_losses = np.minimum.accumulate(all_losses)

        # Calculate distance from the minimum loss achieved
        min_loss = np.min(best_losses)
        distance_from_min = best_losses - min_loss + 1e-8  # Add small epsilon to avoid log(0)

        ax.plot(range(len(distance_from_min)), distance_from_min,
                '-', color=case['color'], label=case['name'],
                linewidth=2, alpha=0.8)

    ax.set_xlabel('Total Evaluations', fontsize=11)
    ax.set_ylabel('Distance from Best Loss', fontsize=11)
    ax.set_title('Loss Convergence During Optimization', fontsize=12, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    # Other subplots: Best parameter values evolution
    for param_idx, param_name in enumerate(param_names):
        ax = axes.flatten()[param_idx + 1]

        for results, case in zip(results_list, test_cases):
            history = results['history']
            all_params = np.array(history['all_X_original'])  # Convert to numpy
            all_losses = np.array(history['all_y'])  # Convert to numpy
            true_value = float(case['true_params'][param_idx])

            # Track best parameters (corresponding to best loss) over time
            # Using numpy vectorized operations
            best_idx_so_far = np.zeros(len(all_losses), dtype=int)
            current_best_idx = 0

            for i in range(len(all_losses)):
                if all_losses[i] < all_losses[current_best_idx]:
                    current_best_idx = i
                best_idx_so_far[i] = current_best_idx

            best_params_history = all_params[best_idx_so_far, param_idx]

            # Plot best parameter evolution
            ax.plot(range(len(best_params_history)), best_params_history,
                    '-', color=case['color'], label=case['name'],
                    linewidth=2, alpha=0.8)

            # Plot true value
            ax.axhline(y=true_value, color=case['color'],
                       linestyle='--', alpha=0.5, linewidth=1)

        ax.set_xlabel('Total Evaluations', fontsize=11)
        ax.set_ylabel(param_name, fontsize=11)
        ax.set_title(f'{param_name} Evolution', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.suptitle('BO-LEAP Detector Calibration', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../plots/detector_calibration_boleap_params.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def print_summary_table(results_list, test_cases):
    """Print a summary table of all results."""

    print("\n" + "=" * 100)
    print("SUMMARY OF BO-LEAP CALIBRATION RESULTS")
    print("=" * 100)
    print(
        f"{'Case':<30} {'MSE':<12} {'Final Loss':<12} {'Scatter Err':<12} {'Reflect Err':<12} {'Absorb Err':<12} {'Runtime (s)':<12}")
    print("-" * 100)

    for case, results in zip(test_cases, results_list):
        name = case['name']
        mse = results['mse']
        final_loss = results['final_loss']
        errors = results['param_errors']
        runtime = results['runtime']

        print(
            f"{name:<30} {mse:<12.6f} {final_loss:<12.6f} {errors[0]:<12.4f} {errors[1]:<12.4f} {errors[2]:<12.4f} {runtime:<12.2f}")

    print("=" * 100)

## Main Execution

In [ ]:
def run_multi_case_calibration_boleap():
    """Run calibration for all test cases using BO-LEAP."""

    print("=" * 60)
    print("DETECTOR CALIBRATION WITH BO-LEAP OPTIMIZER")
    print("=" * 60)
    print(f"\nConfiguration:")
    print(f"  Photons: {Nphot:,}")
    print(f"  K parameter: {K}")
    print(f"  True tau_gs: {TRUE_TAU_GS}")
    print(f"  Simulation tau_gs: {SIM_TAU_GS}")
    print(f"  Loss tau: {LOSS_TAU}")
    print(f"  Lambda (charge, time, intensity): ({LAMBDA_CHARGE}, {LAMBDA_TIME}, {LAMBDA_INTENSITY})")
    print(f"\nBO-LEAP Parameters:")
    print(f"  Iterations: {BO_LEAP_ITERATIONS}")
    print(f"  Local steps: {BO_LEAP_LOCAL_STEPS}")
    print(f"  Population size (K): {BO_LEAP_K}")
    print(f"  Gradient steps (J): {BO_LEAP_J}")
    print(f"  Step size (α): {BO_LEAP_ALPHA}")
    print(f"  GP subset (M): {BO_LEAP_M}")

    # Create test cases
    test_cases = create_test_cases()

    results_list = []

    for case in test_cases:
        # Generate random source and key for this case
        key = jax.random.PRNGKey(int(time.time() * 1000) % 2 ** 32)
        key_source, key_data = jax.random.split(key)

        # Generate source parameters
        source_origin = generate_random_point_inside_cylinder(key_source, r=2, h=4)
        source_intensity = 10000
        source_params = (source_origin, source_intensity)

        # Generate true data with true parameters and TRUE_TAU_GS
        true_detector_params = jnp.concatenate([case['true_params'], jnp.array([TRUE_TAU_GS])])
        true_data = jax.lax.stop_gradient(simulate_event(source_params, true_detector_params, key_data))

        # Run calibration with BO-LEAP
        results = run_single_calibration_boleap(
            case['true_params'],
            case['initial_guess'],
            source_params,
            true_data,
            key_data,
            case['name']
        )

        results_list.append(results)

    # Print summary table
    print_summary_table(results_list, test_cases)

    # Plot results
    plot_boleap_loss_progression(results_list, test_cases)
    plot_parameter_evolution_boleap(results_list, test_cases)

    # Print iteration breakdown
    print("\n" + "=" * 60)
    print("ITERATION BREAKDOWN")
    print("=" * 60)
    for case, results in zip(test_cases, results_list):
        print(f"\n{case['name']}:")
        history = results['history']
        for iter_data in history['iterations']:
            print(f"  Iter {iter_data['iteration'] + 1}: "
                  f"{iter_data['n_population']} population + "
                  f"{iter_data['n_gradient_valid']}/{iter_data['n_gradient_total']} gradient points")
        print(f"  Total evaluations: {history['n_evaluations']}")

    return results_list, test_cases

In [ ]:
# Run the calibration
results_list, test_cases = run_multi_case_calibration_boleap()